In [6]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph , START , END

class State(TypedDict):
    text: str


In [7]:
def greet(state:State):
    return {"text": "Hello! How can I assist you today?"+state["text"]}

def shout(state:State):
    return {"text": state["text"].upper()}

In [8]:
builder = StateGraph(State)
builder.add_node( "greet" , greet)
builder.add_node( "shout" , shout)
builder.add_edge(START, "greet")
builder.add_edge("greet", "shout")
builder.add_edge("shout", END)


start->edge->greet->edge->shout->end

In [11]:
graph = builder.compile()
print(graph.invoke({"text":"Kavish"}))

{'text': 'HELLO! HOW CAN I ASSIST YOU TODAY?KAVISH'}


In [17]:
def append_messages(old:list , new:list)->list:
    return old + new

In [18]:
from typing import Annotated

class ChatState(TypedDict):
   messages: Annotated[list, append_messages]

In [19]:
def echo(state:ChatState):
    last = state["messages"][-1]["content"]
    return {"messages": [{"role": "assistant", "content": "You said: " + last}]}

In [29]:
print(echo({"messages":[{"role":"user","content":"Hello"}]}))

{'messages': [{'role': 'assistant', 'content': 'You said: Hello'}]}


PLUG in llm

In [36]:
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:openai/gpt-oss-20b")


In [38]:
def ChatBot(state: ChatState):
    reply = model.invoke(state["messages"])
    return {"message":[{"role":"assistant","content":reply.content}]}

In [39]:
builder = StateGraph(ChatState)
builder.add_node("chatbot", ChatBot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)
graph = builder.compile()

In [ ]:
out = graph.invoke({"messages": [{"role": "user", "content": "Hi, I'm Kavish"}]})
print(out["messages"][-1]["content"])


Hi, I'm Kavish


In [ ]:
s = input("Enter your message: ")
print(s)


In [ ]:
history = []

while True:
    user_input = input("You: ")
    if user_input.lower() in ["quit", "exit"]:
        break

    history.append({"role": "user", "content": user_input})
    result = graph.invoke({"messages": history})
    history = result["messages"]          # save the full state back

    print("Bot:", history[-1]["content"])
